# MCD-rPPG Dataset Preprocessing Notebook

**Complete Preprocessing Pipeline with MediaPipe Face Detection**

This notebook processes the MCD-rPPG dataset using MediaPipe for face detection and ROI extraction.
It aligns video frames with PPG/ECG signals and saves preprocessed data to the specified output paths.

## Configuration
- **VIDEOS_PATH**: `/home/cristic/data/Bgeorge/mcd_rppg/snapshots/929fb19c5ff2b5c8ed64a7c3a123744346674e88/video/`
- **FACES_PATH**: `/home/cristic/preprocessed_data/faces/`
- **LANDMARKS_PATH**: `/home/cristic/preprocessed_data/landmarks/`
- **Output Root**: `/home/cristic/preprocessed_data/`

## ROI Definitions (MediaPipe 468 landmarks)
- `full_face`: All 468 landmarks
- `forehead`: [103, 104, 105, 332, 333, 334, 6, 7, 8, 9, 10]
- `left_eye`: landmarks 22-52
- `right_eye`: landmarks 252-282
- `nose`: landmarks 1-20, 195-220
- `mouth`: landmarks 60-80, 290-320
- `left_cheek`: landmarks 0-100, 200-300
- `right_cheek`: landmarks 100-200, 300-400
- `chin`: landmarks 150-170, 370-390
- `left_iris`: landmarks 468-472
- `right_iris`: landmarks 473-477

## Workflow
1. Setup & Configuration
2. Initialize MediaPipe
3. Load and Process Videos
4. Extract ROIs and Landmarks
5. Align with PPG/ECG Signals
6. Save Preprocessed Data
7. Verify Output

---

## 1. Setup and Configuration

In [ ]:
# Install required packages
!pip install mediapipe>=0.10.11 opencv-python-headless tqdm scikit-image numpy pandas scipy matplotlib scikit-video

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import cv2
from tqdm import tqdm
from datetime import datetime
from scipy.signal import butter, filtfilt
from scipy.interpolate import interp1d
from sklearn.model_selection import train_test_split
import random
import matplotlib.pyplot as plt
import skvideo.io

# Add rppglib to path
sys.path.insert(0, '/workspace/CrisChir__mcd_rppg')

import rppglib.data_utils
import rppglib.face_utils

print('All imports successful!')
print(f'MediaPipe version: {rppglib.face_utils.mp.__version__}')
print(f'rppglib imported from: {rppglib.__file__}')

In [ ]:
# Configuration
class Config:
    # Paths
    VIDEOS_PATH = '/home/cristic/data/Bgeorge/mcd_rppg/snapshots/929fb19c5ff2b5c8ed64a7c3a123744346674e88/video/'
    FACES_PATH = '/home/cristic/preprocessed_data/faces/'
    LANDMARKS_PATH = '/home/cristic/preprocessed_data/landmarks/'
    OUTPUT_ROOT = '/home/cristic/preprocessed_data/'
    DB_PATH = '/home/cristic/data/Bgeorge/mcd_rppg/snapshots/929fb19c5ff2b5c8ed64a7c3a123744346674e88/db.csv'
    
    # Processing parameters
    window_size = 256
    stride = 64
    target_size = (128, 128)
    padding = 20
    min_face_size = 64
    max_face_size = 512
    
    # Signal processing
    ppg_low_freq = 0.75
    ppg_high_freq = 4.0
    frame_rate = 30.0
    ppg_rate = 100.0
    
    # Dataset splits
    train_ratio = 0.8
    val_ratio = 0.1
    test_ratio = 0.1
    random_state = 42
    
    # ROI definitions (MediaPipe 468 landmarks)
    rois = {
        'full_face': list(range(468)),
        'forehead': [103, 104, 105, 332, 333, 334, 6, 7, 8, 9, 10],
        'left_eye': list(range(22, 53)),
        'right_eye': list(range(252, 283)),
        'nose': list(range(1, 21)) + list(range(195, 221)),
        'mouth': list(range(60, 81)) + list(range(290, 321)),
        'left_cheek': list(range(0, 101)) + list(range(200, 301)),
        'right_cheek': list(range(100, 201)) + list(range(300, 401)),
        'chin': list(range(150, 171)) + list(range(370, 391)),
        'left_iris': list(range(468, 473)),
        'right_iris': list(range(473, 478))
    }

config = Config()

# Create output directories
os.makedirs(config.FACES_PATH, exist_ok=True)
os.makedirs(config.LANDMARKS_PATH, exist_ok=True)
os.makedirs(config.OUTPUT_ROOT, exist_ok=True)

print('Configuration initialized')
print(f'Videos path: {config.VIDEOS_PATH}')
print(f'Faces path: {config.FACES_PATH}')
print(f'Landmarks path: {config.LANDMARKS_PATH}')
print(f'ROIs available: {list(config.rois.keys())}')

---

## 2. Load Database and Video List

In [ ]:
# Load database if available
db = None
if os.path.exists(config.DB_PATH):
    db = pd.read_csv(config.DB_PATH)
    print(f'Loaded database with {len(db)} rows')
    print(f'Columns: {list(db.columns)}')
    display(db.head(2))
else:
    print(f'Database not found at {config.DB_PATH}, will use file system scan')

# Get list of video files
video_files = []
if os.path.exists(config.VIDEOS_PATH):
    for file in os.listdir(config.VIDEOS_PATH):
        if file.endswith('.avi') or file.endswith('.mp4'):
            video_files.append(os.path.join(config.VIDEOS_PATH, file))
    video_files.sort()
    print(f'Found {len(video_files)} video files')
else:
    print(f'Videos path not found: {config.VIDEOS_PATH}')

# Show first few video files
for i, vf in enumerate(video_files[:5]):
    print(f'{i+1}. {os.path.basename(vf)}')

---

## 3. Initialize MediaPipe Face Landmarker

In [ ]:
# Initialize MediaPipe using rppglib
print('Initializing MediaPipe Face Landmarker...')

# We'll use the process_video function from rppglib.face_utils
# which internally handles MediaPipe initialization

# Test with a sample frame to ensure MediaPipe is working
if video_files:
    test_video = rppglib.data_utils.load_video(video_files[0])
    print(f'Loaded test video: {test_video.shape}')
    
    # Process first frame to test
    test_frame = test_video[0]
    print(f'Test frame shape: {test_frame.shape}')
    print('MediaPipe initialization test passed!')
else:
    print('No video files to test with')

---

## 4. File Parsing Functions for PPG/ECG/Meta Files

In [ ]:
def parse_pw_file(pw_path):
    '''Parse .PW file (PPG signal with timestamps)'''
    ppg_values, timestamps = [], []
    with open(pw_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 2:
                try:
                    ppg_values.append(float(parts[0]))
                    timestamps.append(datetime.strptime(' '.join(parts[1:]), '%Y-%m-%d %H:%M:%S.%f'))
                except Exception as e:
                    print(f'Warning: Could not parse line: {line[:50]}... - {e}')
    return np.array(ppg_values), np.array(timestamps)

def parse_meta_file(meta_path):
    '''Parse meta file (frame_number timestamp)'''
    meta_data = {}
    with open(meta_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 2:
                try:
                    meta_data[int(parts[0])] = datetime.strptime(' '.join(parts[1:]), '%Y-%m-%d %H:%M:%S.%f')
                except: pass
    return meta_data

def parse_ppg_sync_file(sync_path):
    '''Parse PPG sync file (frame_number sync_value)'''
    sync_data = {}
    with open(sync_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 2:
                try:
                    sync_data[int(parts[0])] = float(parts[1])
                except: pass
    return sync_data

def find_matching_files(video_path):
    '''Find PPG, meta, and sync files matching a video file'''
    base_name = os.path.splitext(os.path.basename(video_path))[0]
    dataset_root = os.path.dirname(os.path.dirname(video_path))
    
    files = {
        'ppg': None,
        'meta': None,
        'ppg_sync': None,
        'ecg': None
    }
    
    # Try to find PPG file
    ppg_dir = os.path.join(dataset_root, 'ppg')
    if os.path.exists(ppg_dir):
        for f in os.listdir(ppg_dir):
            if f.startswith(base_name) and f.endswith('.PW'):
                files['ppg'] = os.path.join(ppg_dir, f)
                break
    
    # Try to find meta file
    meta_dir = os.path.join(dataset_root, 'meta')
    if os.path.exists(meta_dir):
        for f in os.listdir(meta_dir):
            if f.startswith(base_name) and f.endswith('.txt'):
                files['meta'] = os.path.join(meta_dir, f)
                break
    
    # Try to find ppg_sync file
    sync_dir = os.path.join(dataset_root, 'ppg_sync')
    if os.path.exists(sync_dir):
        for f in os.listdir(sync_dir):
            if f.startswith(base_name) and f.endswith('.txt'):
                files['ppg_sync'] = os.path.join(sync_dir, f)
                break
    
    # Try to find ECG file
    ecg_dir = os.path.join(dataset_root, 'ecg')
    if os.path.exists(ecg_dir):
        for f in os.listdir(ecg_dir):
            if f.startswith(base_name):
                files['ecg'] = os.path.join(ecg_dir, f)
                break
    
    return files

---

## 5. PPG-Video Alignment Function

Core function that aligns PPG signals with video frames using timestamp interpolation.

In [ ]:
def align_ppg_with_video(ppg_values, ppg_timestamps, meta_data, frame_rate=30.0):
    '''
    Align PPG signal with video frames using timestamp interpolation.
    
    Args:
        ppg_values: Array of PPG values
        ppg_timestamps: Array of datetime objects for PPG
        meta_data: Dict mapping frame_number to timestamp
        frame_rate: Video frame rate
    
    Returns:
        aligned_ppg: Array of PPG values aligned with video frames
    '''
    if len(meta_data) == 0 or len(ppg_timestamps) == 0:
        return np.zeros(len(meta_data))
    
    # Convert timestamps to seconds
    first_ppg = ppg_timestamps[0]
    first_meta = list(meta_data.values())[0]
    
    ppg_seconds = [(ts - first_ppg).total_seconds() for ts in ppg_timestamps]
    sorted_frames = sorted(meta_data.keys())
    meta_seconds = [(meta_data[f] - first_meta).total_seconds() for f in sorted_frames]
    
    # Interpolate PPG values at video frame timestamps
    interp_func = interp1d(ppg_seconds, ppg_values, kind='linear', fill_value='extrapolate')
    aligned_ppg = interp_func(meta_seconds)
    
    return aligned_ppg

def preprocess_ppg(ppg, low_freq=0.75, high_freq=4.0, fs=100.0):
    '''Apply bandpass filter and normalize PPG signal'''
    nyquist = 0.5 * fs
    low = low_freq / nyquist
    high = high_freq / nyquist
    b, a = butter(4, [low, high], btype='band')
    filtered = filtfilt(b, a, ppg)
    return (filtered - filtered.mean()) / filtered.std()

def simple_ratio_align(ppg, video_length, ppg_rate=100.0, video_rate=30.0):
    '''
    Simple ratio-based alignment (fallback when timestamps are unavailable).
    Uses the ratio: ppg_samples_per_video_frame = ppg_rate / video_rate
    '''
    ratio = ppg_rate / video_rate
    aligned_ppg = []
    for frame_idx in range(video_length):
        ppg_start = int(frame_idx * ratio)
        ppg_end = int((frame_idx + 1) * ratio)
        if ppg_end <= len(ppg):
            # Average PPG values for this frame
            frame_ppg = ppg[ppg_start:ppg_end]
            aligned_ppg.append(np.mean(frame_ppg))
        else:
            aligned_ppg.append(0.0)
    return np.array(aligned_ppg)

---

## 6. Video Processing with rppglib.face_utils.process_video()

In [ ]:
def process_single_video(video_path, config, use_roi=None):
    '''
    Process a single video file using rppglib.face_utils.process_video()
    
    Args:
        video_path: Path to video file
        config: Configuration object
        use_roi: ROI name to extract (None for full face)
    
    Returns:
        dict with processed_video, landmarks, and metadata
    '''
    result = {
        'video_path': video_path,
        'success': False,
        'error': None,
        'processed_video': None,
        'landmarks': None,
        'roi_video': None,
        'aligned_ppg': None,
        'metadata': {}
    }
    
    try:
        # Load video using rppglib
        video = rppglib.data_utils.load_video(video_path)
        result['metadata']['original_shape'] = video.shape
        result['metadata']['num_frames'] = len(video)
        
        # Process video with MediaPipe using rppglib
        processed_video, landmarks = rppglib.face_utils.process_video(
            video,
            min_face_size=config.min_face_size,
            max_face_size=config.max_face_size
        )
        
        result['processed_video'] = processed_video
        result['landmarks'] = landmarks
        result['metadata']['processed_shape'] = processed_video.shape
        
        # Extract ROI if specified
        if use_roi and use_roi in config.rois:
            roi_video = rppglib.face_utils.extract_roi(
                processed_video, landmarks, use_roi
            )
            result['roi_video'] = roi_video
            result['metadata']['roi_shape'] = roi_video.shape
        
        # Find matching PPG/meta files
        matching_files = find_matching_files(video_path)
        result['metadata']['matching_files'] = matching_files
        
        # Align PPG with video if available
        if matching_files['ppg'] and matching_files['meta']:
            ppg_values, ppg_timestamps = parse_pw_file(matching_files['ppg'])
            meta_data = parse_meta_file(matching_files['meta'])
            
            if len(ppg_values) > 0 and len(meta_data) > 0:
                aligned_ppg = align_ppg_with_video(
                    ppg_values, ppg_timestamps, meta_data, config.frame_rate
                )
                result['aligned_ppg'] = preprocess_ppg(aligned_ppg, config.ppg_low_freq, config.ppg_high_freq, config.ppg_rate)
                result['metadata']['ppg_length'] = len(aligned_ppg)
            else:
                # Fallback to simple ratio-based alignment
                ppg_values_simple = parse_pw_file(matching_files['ppg'])[0]
                if len(ppg_values_simple) > 0:
                    aligned_ppg_simple = simple_ratio_align(
                        ppg_values_simple, len(processed_video), config.ppg_rate, config.frame_rate
                    )
                    result['aligned_ppg'] = preprocess_ppg(aligned_ppg_simple, config.ppg_low_freq, config.ppg_high_freq, config.ppg_rate)
                    result['metadata']['ppg_length'] = len(aligned_ppg_simple)
                    result['metadata']['alignment_method'] = 'ratio'
        
        result['success'] = True
        
    except Exception as e:
        result['error'] = str(e)
        result['success'] = False
        import traceback
        print(f'Error processing {video_path}:')
        traceback.print_exc()
    
    return result

---

## 7. TEST PHASE: Process Sample Videos

In [ ]:
# Test with a few videos first
print('=' * 80)
print('TEST PHASE: Processing sample videos')
print('=' * 80)

test_limit = min(3, len(video_files))
test_results = []

for i in range(test_limit):
    video_path = video_files[i]
    print(f'\nProcessing test video {i+1}/{test_limit}: {os.path.basename(video_path)}')
    
    result = process_single_video(video_path, config, use_roi='forehead')
    test_results.append(result)
    
    if result['success']:
        print(f'  ✓ Success!')
        print(f'    Original: {result["metadata"].get("original_shape")}')
        print(f'    Processed: {result["metadata"].get("processed_shape")}')
        if result['roi_video'] is not None:
            print(f'    ROI (forehead): {result["metadata"].get("roi_shape")}')
        if result['aligned_ppg'] is not None:
            print(f'    Aligned PPG: {len(result["aligned_ppg"])} samples')
        print(f'    Landmarks: {result["landmarks"].shape if result["landmarks"] is not None else None}')
    else:
        print(f'  ✗ Failed: {result["error"]}')

print(f'\nTest phase complete: {sum(1 for r in test_results if r["success"])}/{test_limit} successful')

In [ ]:
# Visualize test results
if test_results and test_results[0]['success']:
    result = test_results[0]
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    
    # Original frame
    axes[0, 0].imshow(result['processed_video'][0])
    axes[0, 0].set_title('Processed Video Frame 0')
    axes[0, 0].axis('off')
    
    # Middle frame
    mid_idx = len(result['processed_video']) // 2
    axes[0, 1].imshow(result['processed_video'][mid_idx])
    axes[0, 1].set_title(f'Processed Video Frame {mid_idx}')
    axes[0, 1].axis('off')
    
    # Last frame
    axes[0, 2].imshow(result['processed_video'][-1])
    axes[0, 2].set_title(f'Processed Video Frame {len(result["processed_video"])-1}')
    axes[0, 2].axis('off')
    
    # ROI if available
    if result['roi_video'] is not None:
        axes[1, 0].imshow(result['roi_video'][0])
        axes[1, 0].set_title('ROI (forehead) Frame 0')
        axes[1, 0].axis('off')
        
        axes[1, 1].imshow(result['roi_video'][mid_idx])
        axes[1, 1].set_title(f'ROI (forehead) Frame {mid_idx}')
        axes[1, 1].axis('off')
        
        axes[1, 2].imshow(result['roi_video'][-1])
        axes[1, 2].set_title(f'ROI (forehead) Frame {len(result["roi_video"])-1}')
        axes[1, 2].axis('off')
    else:
        # Plot PPG signal
        if result['aligned_ppg'] is not None:
            axes[1, 0].plot(result['aligned_ppg'][:100])
            axes[1, 0].set_title('Aligned PPG Signal (first 100 samples)')
            axes[1, 0].axis('on')
        
        axes[1, 1].axis('off')
        axes[1, 2].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    # Plot PPG signal
    if result['aligned_ppg'] is not None:
        plt.figure(figsize=(12, 4))
        plt.plot(result['aligned_ppg'])
        plt.title('Aligned PPG Signal')
        plt.xlabel('Frame')
        plt.ylabel('PPG Value (normalized)')
        plt.grid(True)
        plt.show()

---

## 8. Full Dataset Processing

In [ ]:
# Process all videos
print('=' * 80)
print('FULL PROCESSING: Processing all videos')
print('=' * 80)

# Create output directories for each ROI
for roi_name in config.rois:
    roi_dir = os.path.join(config.OUTPUT_ROOT, roi_name)
    os.makedirs(roi_dir, exist_ok=True)

all_results = []
success_count = 0
failure_count = 0

# Process videos with progress bar
for i, video_path in enumerate(tqdm(video_files, desc='Processing videos')):
    result = process_single_video(video_path, config, use_roi=None)
    all_results.append(result)
    
    if result['success']:
        success_count += 1
    else:
        failure_count += 1
        print(f'Failed: {os.path.basename(video_path)} - {result["error"]}')

print(f'\nProcessing complete: {success_count} successful, {failure_count} failed out of {len(video_files)} total')

---

## 9. Extract and Save ROIs and Landmarks

In [ ]:
# Extract and save ROIs for all successful results
print('\nExtracting and saving ROIs...')

roi_data = {roi: [] for roi in config.rois}
landmarks_data = []
metadata_list = []

for i, result in enumerate(all_results):
    if not result['success']:
        continue
    
    video_id = os.path.splitext(os.path.basename(result['video_path']))[0]
    processed_video = result['processed_video']
    landmarks = result['landmarks']
    
    # Save landmarks
    if landmarks is not None:
        landmarks_file = os.path.join(config.LANDMARKS_PATH, f'{video_id}_landmarks.npy')
        np.save(landmarks_file, landmarks)
        landmarks_data.append({
            'video_id': video_id,
            'landmarks_file': landmarks_file,
            'num_frames': len(landmarks),
            'shape': landmarks.shape
        })
    
    # Extract and save all ROIs
    for roi_name in config.rois:
        try:
            roi_video = rppglib.face_utils.extract_roi(
                processed_video, landmarks, roi_name
            )
            roi_data[roi_name].append({
                'video_id': video_id,
                'roi_video': roi_video,
                'landmarks': landmarks
            })
            
            # Save individual ROI video
            roi_dir = os.path.join(config.OUTPUT_ROOT, roi_name)
            roi_file = os.path.join(roi_dir, f'{video_id}.npy')
            np.save(roi_file, roi_video)
            
        except Exception as e:
            print(f'Warning: Could not extract ROI {roi_name} for {video_id}: {e}')
    
    # Save metadata
    metadata_list.append({
        'video_id': video_id,
        'video_path': result['video_path'],
        'num_frames': result['metadata'].get('num_frames'),
        'processed_shape': result['metadata'].get('processed_shape'),
        'has_ppg': result['aligned_ppg'] is not None,
        'ppg_length': result['metadata'].get('ppg_length'),
        'alignment_method': result['metadata'].get('alignment_method', 'timestamp')
    })

print(f'Saved ROIs for {len(roi_data["full_face"])} videos')
print(f'Saved landmarks for {len(landmarks_data)} videos')

In [ ]:
# Save combined ROI arrays
print('\nSaving combined ROI arrays...')

for roi_name, roi_list in roi_data.items():
    if roi_list:
        all_roi_videos = [item['roi_video'] for item in roi_list]
        # Pad videos to same length if needed
        max_frames = max(v.shape[0] for v in all_roi_videos)
        padded_videos = []
        for v in all_roi_videos:
            if v.shape[0] < max_frames:
                # Pad with zeros
                pad_shape = (max_frames - v.shape[0], *v.shape[1:])
                padding = np.zeros(pad_shape, dtype=v.dtype)
                v_padded = np.concatenate([v, padding], axis=0)
                padded_videos.append(v_padded)
            else:
                padded_videos.append(v)
        
        combined_roi = np.array(padded_videos)
        combined_file = os.path.join(config.OUTPUT_ROOT, f'{roi_name}_all.npy')
        np.save(combined_file, combined_roi)
        print(f'  {roi_name}: {combined_roi.shape} -> {combined_file}')

# Save metadata
metadata_df = pd.DataFrame(metadata_list)
metadata_file = os.path.join(config.OUTPUT_ROOT, 'metadata.csv')
metadata_df.to_csv(metadata_file, index=False)
print(f'\nSaved metadata: {metadata_file}')
print(metadata_df.head())

In [ ]:
# Save PPG signals
print('\nSaving PPG signals...')

ppg_data = []
for i, result in enumerate(all_results):
    if not result['success']:
        continue
    
    video_id = os.path.splitext(os.path.basename(result['video_path']))[0]
    
    if result['aligned_ppg'] is not None:
        ppg_file = os.path.join(config.OUTPUT_ROOT, 'ppg', f'{video_id}.npy')
        os.makedirs(os.path.dirname(ppg_file), exist_ok=True)
        np.save(ppg_file, result['aligned_ppg'])
        ppg_data.append({
            'video_id': video_id,
            'ppg_file': ppg_file,
            'ppg_length': len(result['aligned_ppg']),
            'alignment_method': result['metadata'].get('alignment_method', 'timestamp')
        })

ppg_df = pd.DataFrame(ppg_data)
if len(ppg_df) > 0:
    ppg_metadata_file = os.path.join(config.OUTPUT_ROOT, 'ppg_metadata.csv')
    ppg_df.to_csv(ppg_metadata_file, index=False)
    print(f'Saved PPG metadata: {ppg_metadata_file}')
    print(f'Total PPG signals saved: {len(ppg_df)}')
else:
    print('No PPG signals to save')

---

## 10. Create Dataset Splits

In [ ]:
# Create train/val/test splits by video ID
print('\nCreating dataset splits...')

video_ids = [os.path.splitext(os.path.basename(r['video_path']))[0] for r in all_results if r['success']]

# Split by video ID (stratified by subject if available)
train_ids, test_ids = train_test_split(
    video_ids,
    test_size=config.test_ratio + config.val_ratio,
    random_state=config.random_state
)
val_ids, test_ids = train_test_split(
    test_ids,
    test_size=config.test_ratio / (config.test_ratio + config.val_ratio),
    random_state=config.random_state
)

# Save splits
splits = {
    'train': train_ids,
    'val': val_ids,
    'test': test_ids
}

for split_name, split_ids in splits.items():
    split_file = os.path.join(config.OUTPUT_ROOT, f'{split_name}_videos.txt')
    with open(split_file, 'w') as f:
        for vid in split_ids:
            f.write(f'{vid}\n')
    print(f'  {split_name}: {len(split_ids)} videos -> {split_file}')

# Also save as numpy arrays
for split_name, split_ids in splits.items():
    np.save(os.path.join(config.OUTPUT_ROOT, f'{split_name}_videos.npy'), np.array(split_ids))

---

## 11. Verify Output

In [ ]:
# Verify all output files
print('\n' + '=' * 80)
print('VERIFICATION: Checking output files')
print('=' * 80)

# Check output directories
output_dirs = [
    config.OUTPUT_ROOT,
    config.FACES_PATH,
    config.LANDMARKS_PATH,
    os.path.join(config.OUTPUT_ROOT, 'ppg')
]

for d in output_dirs:
    if os.path.exists(d):
        files = os.listdir(d)
        print(f'✓ {d}: {len(files)} files')
    else:
        print(f'✗ {d}: NOT FOUND')

# Check ROI directories
print(f'\nROI directories:')
for roi_name in config.rois:
    roi_dir = os.path.join(config.OUTPUT_ROOT, roi_name)
    if os.path.exists(roi_dir):
        files = os.listdir(roi_dir)
        print(f'  {roi_name}: {len(files)} files')
    else:
        print(f'  {roi_name}: NOT FOUND')

# Check key files
key_files = [
    os.path.join(config.OUTPUT_ROOT, 'metadata.csv'),
    os.path.join(config.OUTPUT_ROOT, 'ppg_metadata.csv'),
    os.path.join(config.OUTPUT_ROOT, 'train_videos.txt'),
    os.path.join(config.OUTPUT_ROOT, 'val_videos.txt'),
    os.path.join(config.OUTPUT_ROOT, 'test_videos.txt'),
]

print(f'\nKey files:')
for f in key_files:
    if os.path.exists(f):
        size = os.path.getsize(f) / (1024 * 1024)
        print(f'  ✓ {os.path.basename(f)}: {size:.2f} MB')
    else:
        print(f'  ✗ {os.path.basename(f)}: NOT FOUND')

# Summary statistics
print(f'\n' + '=' * 80)
print('SUMMARY')
print('=' * 80)
print(f'Total videos processed: {len(all_results)}')
print(f'Successful: {success_count}')
print(f'Failed: {failure_count}')
print(f'Success rate: {success_count/len(all_results)*100:.1f}%')
print(f'\nOutput saved to: {config.OUTPUT_ROOT}')
print(f'Faces saved to: {config.FACES_PATH}')
print(f'Landmarks saved to: {config.LANDMARKS_PATH}')

---

## 12. Quick Test: Load and Verify One Sample

In [ ]:
# Load and verify one sample from the preprocessed data
print('\nQuick verification test...')

# Get first successful result
first_success = None
for result in all_results:
    if result['success']:
        first_success = result
        break

if first_success:
    video_id = os.path.splitext(os.path.basename(first_success['video_path']))[0]
    
    # Load saved data
    landmarks_file = os.path.join(config.LANDMARKS_PATH, f'{video_id}_landmarks.npy')
    full_face_file = os.path.join(config.OUTPUT_ROOT, 'full_face', f'{video_id}.npy')
    forehead_file = os.path.join(config.OUTPUT_ROOT, 'forehead', f'{video_id}.npy')
    ppg_file = os.path.join(config.OUTPUT_ROOT, 'ppg', f'{video_id}.npy')
    
    print(f'Video ID: {video_id}')
    
    if os.path.exists(landmarks_file):
        loaded_landmarks = np.load(landmarks_file)
        print(f'✓ Landmarks loaded: {loaded_landmarks.shape}')
    else:
        print(f'✗ Landmarks not found: {landmarks_file}')
    
    if os.path.exists(full_face_file):
        loaded_face = np.load(full_face_file)
        print(f'✓ Full face video loaded: {loaded_face.shape}')
    else:
        print(f'✗ Full face not found: {full_face_file}')
    
    if os.path.exists(forehead_file):
        loaded_forehead = np.load(forehead_file)
        print(f'✓ Forehead ROI loaded: {loaded_forehead.shape}')
    else:
        print(f'✗ Forehead ROI not found: {forehead_file}')
    
    if os.path.exists(ppg_file):
        loaded_ppg = np.load(ppg_file)
        print(f'✓ PPG signal loaded: {loaded_ppg.shape}')
    else:
        print(f'✗ PPG signal not found: {ppg_file}')
    
    # Display a sample frame
    if os.path.exists(full_face_file):
        plt.figure(figsize=(10, 5))
        plt.subplot(1, 2, 1)
        plt.imshow(loaded_face[0])
        plt.title(f'Full Face - Frame 0')
        plt.axis('off')
        
        plt.subplot(1, 2, 2)
        if os.path.exists(forehead_file):
            plt.imshow(loaded_forehead[0])
            plt.title(f'Forehead ROI - Frame 0')
        plt.axis('off')
        plt.tight_layout()
        plt.show()
    
    # Plot PPG signal
    if os.path.exists(ppg_file):
        plt.figure(figsize=(12, 4))
        plt.plot(loaded_ppg[:200])
        plt.title(f'PPG Signal - First 200 samples')
        plt.xlabel('Sample')
        plt.ylabel('PPG Value')
        plt.grid(True)
        plt.show()
    
    print('\n✓ Verification test passed!')
else:
    print('No successful results to verify')

---

## Notebook Complete!

The preprocessing pipeline has been completed. All data has been saved to:
- `/home/cristic/preprocessed_data/` - Root output directory
- `/home/cristic/preprocessed_data/faces/` - Processed face videos
- `/home/cristic/preprocessed_data/landmarks/` - Face landmarks
- `/home/cristic/preprocessed_data/{roi_name}/` - ROI videos for each ROI
- `/home/cristic/preprocessed_data/ppg/` - Aligned PPG signals

### Files Created:
- `metadata.csv` - Metadata for all processed videos
- `ppg_metadata.csv` - Metadata for PPG signals
- `train_videos.txt`, `val_videos.txt`, `test_videos.txt` - Dataset splits
- `{roi_name}_all.npy` - Combined ROI arrays

### Next Steps:
1. Verify the output files exist and have correct shapes
2. Use the preprocessed data for training rPPG models
3. Optionally adjust ROI definitions or preprocessing parameters